**STEAM MACHINE LEARNING**

**Integrantes**:

João Pedro Costa Cruz

João Vitor Souza Tavares

Lucas Antônio Araújo Santos

**Link para o dataset original**:

https://www.kaggle.com/datasets/fronkongames/steam-games-dataset

**Descrição do problema**:

O dataset escolhido traz informações sobre jogos virtuais da plataforma Steam. A ideia é determinar se um jogo é um "Sucesso". Um jogo é considerado um sucesso se ele tiver uma taxa de aprovação de reviews positivos maior ou igual a 80% e ao menos 100 reviews.

O problema foi estruturado como uma tarefa de **Classificação** porque o nosso objetivo final é mapear os dados de entrada para uma decisão categórica e discreta (Sucesso vs. Não Sucesso). Isso permite que tomadores de decisão (como publicadoras de jogos) façam avaliações qualitativas e utilizem métricas de diagnóstico de erro cruciais (como a taxa de falsos positivos e falsos negativos através de matrizes de confusão) para minimizar os riscos de investimento.

In [ ]:
#==================================
#   IMPORTAÇÃO DAS BIBLIOTECAS
#==================================

# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações do ambiente de plotagem
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Ignorar avisos de atualizações de bibliotecas
import warnings
warnings.filterwarnings('ignore')

print("Configurações do ambiente concluídas.")

In [ ]:
#============================================
#   DATASET E DEFINIÇÃO DO ATRIBUTO-ALVO
#============================================

# URL do dataset
url_dataset = "https://raw.githubusercontent.com/LucasAAraujoS/steam-machine-learning/refs/heads/main/steam_games_clean.csv"

# Carrega o arquivo usando o ponto e vírgula (;) como delimitador
df = pd.read_csv(url_dataset, sep=';')

#----------------------------------------------------------------------------------------------------------------------------

# Cálculo do volume total de avaliações (reviews)
df['Total_Reviews'] = df['Positive'] + df['Negative']

# Cálculo da taxa de aprovação de um jogo
df['Approval_Rate'] = np.where(df['Total_Reviews'] > 0, df['Positive'] / df['Total_Reviews'], 0.0)

# Definição do atributo-alvo
df['Success'] = ((df['Approval_Rate'] >= 0.80) & (df['Total_Reviews'] >= 100)).astype(int)

print(f"Dataset carregado com sucesso!")
print(f"Dimensões do DataFrame: {df.shape[0]} linhas e {df.shape[1]} colunas.")

## 1. Engenharia do Alvo e Estatística Descritiva

Esta seção cobre: distribuição do atributo-alvo (`Success`), estatística descritiva geral do dataset e remoção das colunas usadas para construir o alvo (evitando data leakage).

In [ ]:
#==================================
#   DISTRIBUIÇÃO DO ATRIBUTO-ALVO
#==================================

success_counts = df['Success'].value_counts().sort_index()
success_pct = df['Success'].value_counts(normalize=True).sort_index() * 100

print("Contagem por classe:")
print(success_counts)
print("\nProporção por classe (%):")
print(success_pct.round(2))
print("\n")

fig, ax = plt.subplots()
sns.countplot(x='Success', data=df, palette=['#c0392b', '#27ae60'], ax=ax)
ax.set_xticklabels(['Não Sucesso (0)', 'Sucesso (1)'])
ax.set_title('Distribuição da variável alvo: Sucesso vs. Não Sucesso')
ax.set_xlabel('')
ax.set_ylabel('Quantidade de jogos')
for i, v in enumerate(success_counts):
    ax.text(i, v + 1500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretação:**

O dataset apresenta um forte desbalanceamento de classes: **89,56% dos jogos (112.718) são classificados como "Não Sucesso"**, enquanto apenas **10,44% (13.137) são "Sucesso"**, ou seja, uma proporção aproximada de 9 para 1.

Isso reflete uma realidade do mercado da Steam: a grande maioria dos jogos publicados na plataforma não atinge um patamar consistente de avaliações positivas somado a um volume relevante de reviews — o que é coerente com o fato de a Steam ser uma plataforma aberta, com milhares de lançamentos pequenos, indie ou pouco divulgados.

Esse desbalanceamento é uma informação crítica para as próximas etapas do projeto: ele justifica (a) o uso de `stratify=y` na separação treino/teste (Etapa 2) para preservar essa proporção em ambos os conjuntos, e (b) a priorização do F1-Score e da matriz de confusão em vez da acurácia simples na avaliação dos modelos (Etapa 4), já que um classificador ingênuo que sempre previsse "Não Sucesso" já acertaria quase 90% dos casos sem aprender nada de útil.

### Estatística descritiva geral do dataset

In [ ]:
#==================================
#   ESTATÍSTICA DESCRITIVA GERAL
#==================================

print(f"O dataset possui {df.shape[0]:,} registros e {df.shape[1]} atributos.\n")

print("Tipos de dados por coluna:")
print(df.dtypes)

valores_ausentes = df.isnull().sum()
print("\nValores ausentes por coluna (apenas colunas com ausências):")
print(valores_ausentes[valores_ausentes > 0])

print(f"\nLinhas totalmente duplicadas: {df.duplicated().sum()}")
print(f"Nomes de jogos duplicados (podem ser DLCs/remasters/reentradas): {df['Name'].duplicated().sum()}")

df[['Price', 'Positive', 'Negative', 'Achievements']].describe()

**Interpretação:**

O dataset contém 125.855 jogos e 14 atributos, com um volume ausente concentrado em variáveis textuais/categóricas: `Developers` (8.449), `Publishers` (8.922), `Categories` (8.966) e `Genres` (8.423) — em torno de 6,7% das linhas não têm gênero ou desenvolvedor informado, o que provavelmente reflete jogos cadastrados de forma incompleta na Steam (ex: *playtests* ou títulos removidos). Há também 18 linhas totalmente duplicadas e 1.210 nomes repetidos, que precisarão ser investigados posteriormente (podem ser duplicatas reais ou apenas jogos com o mesmo nome).

Quanto ao preço, `a média (US$ 4,81) é bem maior que a mediana (US$ 2,39)`, o que indica uma distribuição assimétrica à direita: a maioria dos jogos é barata, mas alguns títulos caros (até US$ 999,98) puxam a média para cima. O mesmo padrão, ainda mais extremo, aparece em `Positive` e `Negative`: a mediana de reviews positivas é de apenas 4, mas a média passa de 1.000, evidenciando a presença de outliers fortes — alguns poucos jogos com milhões de reviews (blockbusters) distorcem completamente a média em relação à realidade da maioria dos títulos da plataforma.

### Remoção das colunas usadas para construir o alvo (evitando *data leakage*)

In [ ]:
#===================================================
#   REMOÇÃO DAS COLUNAS DE VAZAMENTO (DATA LEAKAGE)
#===================================================

# Positive, Negative, Total_Reviews e Approval_Rate foram usadas para CALCULAR
# o alvo (Success). Mantê-las como atributos preditivos permitiria ao modelo
# 'colar' a resposta em vez de aprender padrões reais.
#
# Mantemos 'df' completo para a análise exploratória (Membro B e C ainda vão
# usar Positive/Negative/Approval_Rate para investigar padrões) e criamos
# 'df_model' -- sem essas colunas -- para ser o ponto de partida da Etapa 2
# (pré-processamento e modelagem).

leakage_cols = ['Positive', 'Negative', 'Total_Reviews', 'Approval_Rate']

df_model = df.drop(columns=leakage_cols)

print("Colunas removidas para evitar data leakage:", leakage_cols)
print(f"\nColunas restantes para modelagem ({df_model.shape[1]}):")
print(list(df_model.columns))

**Interpretação:**

`Positive`, `Negative`, `Total_Reviews` e `Approval_Rate` não podem seguir para o treinamento porque foram exatamente as variáveis usadas para definir `Success`: um modelo treinado com elas não estaria aprendendo a prever sucesso a partir de características do jogo (preço, gênero, plataforma etc.), e sim reconstruindo uma fórmula que já conhecemos — isso é chamado de vazamento de dados (*data leakage*) e infla artificialmente as métricas sem gerar um modelo útil na prática.

`df_model` guarda a versão do dataset já livre desse problema, pronta para ser usada. `df` (com todas as colunas originais) continua disponível para as análises de correlação e distribuição que vêm a seguir.

# 2. Correlação e Relações com o Alvo.

Esta seção cobre: correlação das variáveis numéricas com `Success`, gráfico de dispersão preço x taxa de aprovação, e tabelas de frequência cruzando plataformas e gêneros com a taxa de sucesso.

### Correlação das variáveis numéricas com o alvo

In [ ]:
#==================================
#   CORRELAÇÃO COM O ALVO
#==================================

# Extrai o ano de lançamento a partir da data (string -> datetime)
df['Release_Year'] = pd.to_datetime(df['Release date'], errors='coerce').dt.year

# Faz a correlação retirando a linha 'sucess', pois a correlação sempre seria 1
num_cols = ['Price', 'Achievements', 'Release_Year']
correlacoes = df[num_cols + ['Success']].corr()['Success'].drop('Success')
print("Correlação (Pearson) com Success:")
print(correlacoes.sort_values(ascending=False))

# Reliza a plotagem do gráfico
fig, ax = plt.subplots()
correlacoes.sort_values().plot(kind='barh', color='#2980b9', ax=ax)
ax.set_title('Correlação das variáveis numéricas com Success')
ax.set_xlabel('Coeficiente de correlação')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

**Interpretação:**

Nenhuma variável numérica isolada tem correlação linear forte com `Success` — o que é esperado, já que sucesso de um jogo é multifatorial. `Price` (0,036) e `Achievements` (0,047) praticamente não têm relação linear direta com o sucesso, sugerindo que preço e quantidade de conquistas, por si só, não determinam a recepção do jogo.

O destaque é `Release_Year`, com correlação negativa de -0,23: jogos mais recentes tendem a ter taxa de sucesso menor. Isso provavelmente não significa que jogos novos são piores, e sim um efeito do próprio critério do alvo (`Total_Reviews >= 100`) — jogos lançados há mais tempo tiveram mais tempo para acumular reviews e ultrapassar esse limiar, enquanto lançamentos recentes ainda estão a caminho. Vale mencionar essa limitação na discussão crítica.

### Gráfico de dispersão: Preço x Taxa de Aprovação

In [ ]:
#==================================
#   DISPERSÃO: PREÇO x APROVAÇÃO
#==================================

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df[df['Total_Reviews'] >= 100],  # foca em jogos com volume relevante de reviews
    x='Price', y='Approval_Rate', hue='Success',
    palette={0: '#c0392b', 1: '#27ae60'}, alpha=0.4, ax=ax
)
ax.set_title('Preço x Taxa de Aprovação (jogos com \u2265 100 reviews)')
ax.set_xlabel('Preço (US$)')
ax.set_ylabel('Taxa de aprovação')
ax.set_xlim(0, 100)  # a maioria dos jogos custa bem menos que isso. Corta outliers extremos de preço
ax.axhline(0.80, color='black', linestyle='--', linewidth=0.8, label='Limiar de Sucesso (80%)')
ax.legend(title='Success')
plt.tight_layout()
plt.show()

**Interpretação:**

O gráfico mostra que jogos de sucesso (verde) aparecem espalhados por praticamente toda a faixa de preço, sem se concentrarem nem nos jogos mais baratos nem nos mais caros — reforçando a baixa correlação encontrada acima. O que fica visível é que jogos com preço próximo de zero têm uma dispersão de aprovação muito maior (do 0% ao 100%), enquanto jogos pagos tendem a se agrupar um pouco mais para cima na taxa de aprovação, possivelmente porque cobrar um preço filtra parte do público e reduz avaliações de "experimentação" impulsivas.

### Tabela de frequência: Plataformas x Taxa de Sucesso

In [ ]:
#==================================
#   PLATAFORMAS x TAXA DE SUCESSO
#==================================

plataformas = ['Windows', 'Mac', 'Linux']
dados_grafico = {}

for plataforma in plataformas:
    taxa = df.groupby(plataforma)['Success'].mean() * 100
    dados_grafico[plataforma] = taxa

# Transforma os resultados em um DataFrame fácil de plotar
df_plataformas = pd.DataFrame(dados_grafico)
df_plataformas.index = ['Não Compatível (0)', 'Compatível (1)']

print("Tabela de taxas de sucesso (%):")
print(df_plataformas.round(2))

# Realiza a plotagem do gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(8, 5))
df_plataformas.T.plot(kind='bar', color=['#e74c3c', '#2ecc71'], ax=ax, edgecolor='black', alpha=0.8)

ax.set_title('Taxa de Sucesso por Compatibilidade de Sistema Operacional')
ax.set_ylabel('Taxa de Sucesso (%)')
ax.set_xlabel('Plataformas')
ax.set_xticklabels(plataformas, rotation=0) # Mantém os nomes Windows/Mac/Linux retos
ax.grid(axis='y', linestyle='--', alpha=0.5) # Linhas de grade ao fundo para ajudar a ler as alturas
ax.legend(title='Suporte')

plt.tight_layout()
plt.show()

**Interpretação:**

Jogos compatíveis com Windows têm taxa de sucesso de 10,44%, contra apenas 2,27% dos que não são compatíveis — mas isso provavelmente reflete que jogos sem suporte a Windows são um grupo muito pequeno e atípico na Steam (plataforma majoritariamente Windows), não necessariamente uma relação causal.

Mais interessante é o padrão em Mac e Linux: jogos compatíveis com Mac têm taxa de sucesso de 19,88% (contra 8,47% sem suporte), e com Linux, 17,88% (contra 9,35%). Isso sugere que oferecer suporte multiplataforma pode estar associado a jogos mais bem cuidados tecnicamente ou voltados a um público mais engajado, embora não devamos interpretar isso como "adicionar suporte a Mac/Linux causa sucesso" sem mais investigação.

### Tabela de frequência: Gêneros x Taxa de Sucesso

In [ ]:
#==================================
#   GÊNEROS x TAXA DE SUCESSO
#==================================

# 'Genres' é multi-valorado (um jogo pode ter vários gêneros separados por vírgula),
# então precisamos 'explodir' a coluna antes de agrupar.
generos = df[['Genres', 'Success']].dropna(subset=['Genres']).copy()
generos['Genres'] = generos['Genres'].str.split(',')
generos = generos.explode('Genres')
generos['Genres'] = generos['Genres'].str.strip()

# Filtra os 10 gêneros mais populares
top_generos = generos['Genres'].value_counts().head(10).index
tabela_generos = (
    generos[generos['Genres'].isin(top_generos)]
    .groupby('Genres')['Success']
    .agg(taxa_sucesso='mean', quantidade='count')
    .sort_values('taxa_sucesso', ascending=False)
)
tabela_generos['taxa_sucesso'] = (tabela_generos['taxa_sucesso'] * 100).round(2)
print(tabela_generos)

# Plotagem do gráfico
fig, ax = plt.subplots()
tabela_generos['taxa_sucesso'].sort_values().plot(kind='barh', color='#8e44ad', ax=ax)
ax.set_title('Taxa de sucesso por gênero (10 gêneros mais frequentes)')
ax.set_xlabel('Taxa de sucesso (%)')
plt.tight_layout()
plt.show()

**Interpretação:**

Entre os 10 gêneros mais frequentes, **RPG** (13,46%), **Adventure** (12,66%) e **Simulation** (11,73%) têm as maiores taxas de sucesso, enquanto **Early Access** (6,16%) e **Free To Play** (8,25%) têm as menores. Faz sentido: jogos em Early Access ainda estão em desenvolvimento e tendem a acumular críticas sobre bugs e conteúdo incompleto, e jogos Free To Play atraem um público muito maior e mais heterogêneo, o que aumenta a chance de reviews negativas de usuários que não são o público-alvo real do jogo. Já **Indie**, apesar de ser o gênero mais frequente do dataset (82.884 jogos), tem taxa de sucesso próxima da mediana (10,84%), mostrando que ser indie não é, por si só, indicativo de melhor ou pior recepção.


# 3. Distribuição das Variáveis Preditivas

Esta seção cobre: histogramas e boxplots das variáveis numéricas preditivas, identificação de outliers, e verificação de inconsistências no dataset.

### Histograma do Preço (Price)

In [ ]:
#==================================
#   DISTRIBUIÇÃO DO PREÇO
#==================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Price'], bins=50, ax=axes[0], color='#2980b9')
axes[0].set_title('Distribuição do Preço (escala normal)')
axes[0].set_xlabel('Preço (US$)')

sns.histplot(df[df['Price'] > 0]['Price'], bins=50, log_scale=True, ax=axes[1], color='#2980b9')
axes[1].set_title('Distribuição do Preço > 0 (escala log)')
axes[1].set_xlabel('Preço (US$, escala log)')

plt.tight_layout()
plt.show()

print(f"Jogos com preço igual a zero (gratuitos): {(df['Price']==0).sum():,} ({(df['Price']==0).mean()*100:.2f}%)")

**Interpretação:**

Na escala normal, a distribuição do preço é extremamente concentrada perto de zero, com uma cauda longa — a visualização em escala log (à direita) é necessária para enxergar a forma real da distribuição entre os jogos pagos, que se aproxima de uma distribuição log-normal, comum em preços de mercado (muitos produtos baratos, poucos caros).

21,18% dos jogos (26.661) são totalmente gratuitos (Price = 0), o que é coerente com o modelo Free To Play amplamente adotado na Steam e reforça por que essa variável sozinha tem baixo poder preditivo isolado (visto nas analises anteriores): jogos gratuitos e pagos aparecem em ambas as classes de sucesso.

### Boxplot do Preço por classe do alvo

In [ ]:
#==================================
#   BOXPLOT: PREÇO x SUCCESS
#==================================

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='Success', y='Price', ax=ax, showfliers=False)
ax.set_xticklabels(['Não Sucesso (0)', 'Sucesso (1)'])
ax.set_title('Preço por classe (outliers extremos ocultados para leitura)')
ax.set_xlabel('')
ax.set_ylabel('Preço (US$)')
plt.tight_layout()
plt.show()

print(df.groupby('Success')['Price'].describe()[['mean','50%','75%','max']])

**Interpretação:**

Jogos de sucesso têm mediana de preço mais alta `(US$ 3,99) do que os de não sucesso (US$ 1,99)`, e o terceiro quartil também é maior `(US$ 8,74 vs. US$ 4,99)`. Isso não contradiz a baixa correlação linear vista nas analises anteriores — a relação parece existir, mas não é linear: jogos de sucesso tendem a se concentrar numa faixa de preço um pouco mais alta, sem que "preço maior" garanta sucesso de forma proporcional.

### Distribuição de Achievements e identificação de outliers (IQR)

In [ ]:
#==================================
#   DISTRIBUIÇÃO DE ACHIEVEMENTS E OUTLIERS (REGRA DO IQR)
#==================================

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df[df['Achievements'] <= 200]['Achievements'], bins=50, color='#e67e22', ax=ax)
ax.set_title('Distribuição de Achievements (até 200, para visualização)')
ax.set_xlabel('Quantidade de conquistas')
plt.tight_layout()
plt.show()

# Identificação de outliers pela regra do IQR (1.5x)
q1, q3 = df['Achievements'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr # Fórmula padrão da estatística

n_outliers = (df['Achievements'] > limite_superior).sum()
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Limite superior (Q3 + 1.5*IQR): {limite_superior}")
print(f"Jogos acima do limite (outliers): {n_outliers:,} ({n_outliers/len(df)*100:.2f}%)")
print(f"Máximo de conquistas no dataset: {df['Achievements'].max():,}")
print(f"Jogos com 0 conquistas: {(df['Achievements']==0).sum():,} ({(df['Achievements']==0).mean()*100:.2f}%)")

**Interpretação:**

Quase metade dos jogos (48,57%) não tem nenhuma conquista cadastrada, e a regra do IQR (Intervalo Interquartil) aponta 8.606 jogos (6,84%) como outliers — jogos com mais de ~48 conquistas, chegando a um máximo de 9.821 (um valor extremo que provavelmente representa um jogo com sistema de conquistas geradas proceduralmente, não um erro de digitação). Como essa variável é fortemente concentrada em zero com uma cauda muito longa, ela também deve ser tratada com cautela (ex: escalonamento robusto a outliers, como `RobustScaler`, em vez de `StandardScaler`).

### Verificação de inconsistências no dataset

In [ ]:
#==================================
#   VERIFICAÇÃO DE INCONSISTÊNCIAS
#==================================

print("Valores de Price negativos:", (df['Price'] < 0).sum())
print("Valores de Achievements negativos:", (df['Achievements'] < 0).sum())

# 'Estimated owners' vem como faixa em texto (ex: '0 - 20000'); não é numérica direto
print("\nExemplos de valores em 'Estimated owners':")
print(df['Estimated owners'].value_counts().head(5))

# Jogos com muitas reviews mas 0 owners estimados (faixa '0 - 0') seriam uma inconsistência
inconsistentes = df[(df['Estimated owners'] == '0 - 0') & (df['Total_Reviews'] > 0)]
print(f"\nJogos na faixa '0 - 0' de owners mas com reviews > 0 (possível inconsistência): {len(inconsistentes):,}")

**Interpretação:**

Não há valores negativos em `Price` ou `Achievements`, e nenhum jogo classificado na faixa "0 - 0" de proprietários estimados possui reviews registradas — bons sinais de integridade básica dos dados, sem inconsistências lógicas entre essas variáveis. Já `Estimated owners` está armazenada como faixas de texto (ex: `"0 - 20000"`), não como número — isso não é um erro, mas uma característica da fonte de dados, e precisa ser tratada (ex: convertida para o ponto médio da faixa) caso seja usada como atributo preditivo posteriormente.

## 4. Pré-processamento e Separação dos Dados

Esta seção cobre: preparação de `X` e `y` a partir de `df_model`, separação treino/teste com estratificação, e codificação das variáveis categóricas de gêneros e plataformas.

### Preparação das Variáveis de entrada

In [ ]:
#==================================
#   IMPORTAÇÕES NECESSÁRIAS PARA ESTA ETAPA
#==================================

# Import para dividir os dados entre treino e teste
from sklearn.model_selection import train_test_split
# Transforma listas múltiplas em colunas numéricas
from sklearn.preprocessing import MultiLabelBinarizer

#==================================
#   PREPARAÇÃO DE X e y
#==================================

# Partimos de df_model (já sem as colunas de vazamento definidas no começo)
dados = df_model.copy()

# 'Name' é um identificador único do jogo, não um atributo preditivo -- não ajuda o
# modelo a generalizar para jogos novos, então é removida aqui.
dados = dados.drop(columns=['Name'])

# 'Release date' é uma string (ex: 'Aug 1, 2023'); extraímos o ano, que é a informação
# temporal mais útil e fácil de tratar numericamente, e descartamos a coluna original.
dados['Release_Year'] = pd.to_datetime(dados['Release date'], errors='coerce').dt.year
dados = dados.drop(columns=['Release date'])

# Alvo
y = dados['Success']
# Variáceis Preditoras
X = dados.drop(columns=['Success'])

print(f"Formato de X: {X.shape}")
print(f"Formato de y: {y.shape}")
print("\nColunas de X:")
print(list(X.columns))

**Interpretação:**

`X` reúne os atributos preditivos definidos no README (Price, Release_Year, Achievements, Windows/Mac/Linux, Developers, Publishers, Categories, Genres), sem `Name` (identificador) e sem as colunas de reviews já removidas na Etapa 1. `y` é a variável-alvo (`Success`). A partir daqui, qualquer tratamento (encoding, escalonamento, imputação) deve ser ajustado *apenas* em cima do conjunto de treino, para não vazar informação do teste.

### Separação treino/teste com estratificação

In [ ]:
#==================================
#   SEPARAÇÃO TREINO/TESTE (80/20, ESTRATIFICADA)
#==================================

# Realização da separação entre treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape[0]:,} jogos ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Teste:  {X_test.shape[0]:,} jogos ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\nProporção da classe Success no conjunto completo:")
print(y.value_counts(normalize=True).round(4) * 100)
print("\nProporção da classe Success no treino:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print("\nProporção da classe Success no teste:")
print(y_test.value_counts(normalize=True).round(4) * 100)

**Interpretação:**

Optamos pela proporção 80/20: com 125.855 jogos no total, mesmo os 20% de teste (aprox. 25.171 jogos) são uma amostra grande o suficiente para uma avaliação final estatisticamente confiável, enquanto os 80% de treino (aprox. 100.684 jogos) dão bastante volume para os modelos aprenderem — especialmente importante aqui, já que a classe "Sucesso" é minoritária (10,44%) e precisa de exemplos suficientes mesmo depois da divisão.

O parâmetro `stratify=y` garante que essa proporção de 89,56%/10,44% entre as classes seja preservada tanto no treino quanto no teste (a saída dos `value_counts` acima confirma isso). Sem estratificação, uma divisão aleatória comum poderia, por azar, concentrar desproporcionalmente os jogos de sucesso em um dos dois conjuntos, distorcendo tanto o treinamento quanto a avaliação dos modelos. O conjunto de teste (`X_test`, `y_test`) fica reservado e não deve ser tocado novamente por algum tempo.

### Codificação de variáveis categóricas: gêneros e categorias (Multi-Label Binarizer)

In [ ]:
#=========================================================
#   CODIFICAÇÃO DE GENRES E CATEGORIES (MULTI-LABEL)
#=========================================================

# 'Genres' e 'Categories' são colunas multi-valoradas (um jogo pode ter vários
# gêneros/categorias separados por vírgula), então One-Hot Encoding tradicional não
# serve -- usamos o MultiLabelBinarizer, que cria uma coluna binária por rótulo
# possível e marca 1 para cada rótulo presente no jogo.

# Jogos sem gênero/categoria informado (valores ausentes) viram uma lista vazia,
# ou seja, um vetor de zeros -- o tratamento mais aprofundado de dados ausentes
# nas demais variáveis fica para as próximas partes

def para_lista(coluna):
    return coluna.fillna('').apply(
        lambda x: [item.strip() for item in x.split(',')] if x != '' else []
    )

genres_train_lista = para_lista(X_train['Genres'])
genres_test_lista = para_lista(X_test['Genres'])
categories_train_lista = para_lista(X_train['Categories'])
categories_test_lista = para_lista(X_test['Categories'])

# O MultiLabelBinarizer é ajustado (fit) SOMENTE no treino, e apenas aplicado
# (transform) no teste
mlb_genres = MultiLabelBinarizer()
genres_train_encoded = mlb_genres.fit_transform(genres_train_lista)
genres_test_encoded = mlb_genres.transform(genres_test_lista)

mlb_categories = MultiLabelBinarizer()
categories_train_encoded = mlb_categories.fit_transform(categories_train_lista)
categories_test_encoded = mlb_categories.transform(categories_test_lista)

genres_train_df = pd.DataFrame(genres_train_encoded, columns=[f'Genre_{g}' for g in mlb_genres.classes_], index=X_train.index)
genres_test_df = pd.DataFrame(genres_test_encoded, columns=[f'Genre_{g}' for g in mlb_genres.classes_], index=X_test.index)
categories_train_df = pd.DataFrame(categories_train_encoded, columns=[f'Cat_{c}' for c in mlb_categories.classes_], index=X_train.index)
categories_test_df = pd.DataFrame(categories_test_encoded, columns=[f'Cat_{c}' for c in mlb_categories.classes_], index=X_test.index)

print(f"Gêneros distintos codificados: {len(mlb_genres.classes_)}")
print(f"Categorias distintas codificadas: {len(mlb_categories.classes_)}")
genres_train_df.head()

**Interpretação:**

O `MultiLabelBinarizer` transformou `Genres` em 33 colunas binárias (uma por gênero, como `Genre_RPG`, `Genre_Indie`) e `Categories` em 58 colunas (como `Cat_Single-player`, `Cat_Multi-player`) — cada jogo passa a ter 1 nas colunas correspondentes aos seus gêneros/categorias reais e 0 nas demais, preservando a informação de que um jogo pode pertencer a vários grupos ao mesmo tempo, o que o One-Hot Encoding tradicional (feito para uma categoria por linha) não conseguiria representar corretamente.

### Codificação de variáveis categóricas: compatibilidade de plataformas

In [ ]:
#==================================
#   CODIFICAÇÃO DAS PLATAFORMAS
#==================================

# Windows, Mac e Linux já são variáveis booleanas (True/False), que já funcionam
# como uma codificação binária -- só convertemos explicitamente para 0/1 (inteiro),
# formato que os classificadores do scikit-learn exigem.

plataformas_train = X_train[['Windows', 'Mac', 'Linux']].astype(int)
plataformas_test = X_test[['Windows', 'Mac', 'Linux']].astype(int)

plataformas_train.head()

**Interpretação:**

Diferente de `Genres` e `Categories`, as plataformas já vinham em três colunas booleanas separadas (uma por sistema operacional), o que já é, na prática, uma codificação binária pronta -- não é necessário aplicar One-Hot Encoding ou Multi-Label Binarizer aqui, apenas garantir o tipo inteiro (0/1) para compatibilidade com os modelos.

### Consolidação das variáveis codificadas

In [ ]:
#===========================================
#   CONSOLIDAÇÃO: VARIÁVEIS JÁ TRATADAS
#===========================================

# Reúne o que já foi processado nesta etapa (plataformas + gêneros + categorias).
# Price, Achievements, Release_Year, Developers, Publishers e Estimated owners
# ainda não foram tratados -- (dados ausentes e escalonamento) decidir, incluindo
# o que fazer com Developers/Publishers, que têm cardinalidade muito alta
# (72.859 e 64.607 valores distintos) para um One-Hot Encoding direto.

X_train_membroA = pd.concat([plataformas_train, genres_train_df, categories_train_df], axis=1)
X_test_membroA = pd.concat([plataformas_test, genres_test_df, categories_test_df], axis=1)

print(f"X_train_membroA: {X_train_membroA.shape}")
print(f"X_test_membroA: {X_test_membroA.shape}")
X_train_membroA.head()

**Interpretação:**

Ao final desta parte, temos `X_train`/`X_test` (divisão bruta, estratificada) e as variáveis categóricas de gêneros, categorias e plataformas já codificadas em `X_train_membroA`/`X_test_membroA` (94 colunas: 3 de plataforma + 33 de gênero + 58 de categoria). As variáveis numéricas (`Price`, `Achievements`, `Release_Year`) e as de alta cardinalidade (`Developers`, `Publishers`, `Estimated owners`) seguem para serem tratadas a seguir, juntamente com valores ausentes e aplicar escalonamento antes da junção final de todas as variáveis para a modelagem.

### Tratamento de Desenvolvedoras e Publicadoras

In [ ]:
#=========================================================
#   TOP N DEVELOPERS E PUBLISHERS + ONE-HOT ENCODING
#=========================================================

TOP_N = 30

# Identifica as Top N empresas no conjunto de treino
top_devs = X_train['Developers'].value_counts().head(TOP_N).index.tolist()
top_pubs = X_train['Publishers'].value_counts().head(TOP_N).index.tolist()

# Função para agrupar em Top N ou 'Outros'
def simplificar_empresa(coluna, top_list):
    coluna_limpa = coluna.fillna('Outros')
    return coluna_limpa.apply(lambda x: x if x in top_list else 'Outros')

# Aplicação no Treino e Teste
X_train_dev_cat = simplificar_empresa(X_train['Developers'], top_devs)
X_test_dev_cat = simplificar_empresa(X_test['Developers'], top_devs)

X_train_pub_cat = simplificar_empresa(X_train['Publishers'], top_pubs)
X_test_pub_cat = simplificar_empresa(X_test['Publishers'], top_pubs)

# One-Hot Encoding para as Top N
devs_train_df = pd.get_dummies(X_train_dev_cat, prefix='Dev', dtype=int)
devs_test_df = pd.get_dummies(X_test_dev_cat, prefix='Dev', dtype=int)

pubs_train_df = pd.get_dummies(X_train_pub_cat, prefix='Pub', dtype=int)
pubs_test_df = pd.get_dummies(X_test_pub_cat, prefix='Pub', dtype=int)

# Alinha as colunas do teste com as do treino
devs_test_df = devs_test_df.reindex(columns=devs_train_df.columns, fill_value=0)
pubs_test_df = pubs_test_df.reindex(columns=pubs_train_df.columns, fill_value=0)

# Remove a coluna genérica 'Outros' para evitar colinearidade estrita
if 'Dev_Outros' in devs_train_df.columns:
    devs_train_df = devs_train_df.drop(columns=['Dev_Outros'])
    devs_test_df = devs_test_df.drop(columns=['Dev_Outros'])

if 'Pub_Outros' in pubs_train_df.columns:
    pubs_train_df = pubs_train_df.drop(columns=['Pub_Outros'])
    pubs_test_df = pubs_test_df.drop(columns=['Pub_Outros'])

print(f"Colunas criadas para Developers: {devs_train_df.shape[1]}")
print(f"Colunas criadas para Publishers: {pubs_train_df.shape[1]}")

**Interpretação:**

As colunas `Developers` e `Publishers` contêm mais de 130.000 nomes distintos no total, a maioria associada a apenas 1 ou 2 jogos (o ecossistema *indie*). Realizar um *One-Hot Encoding* irrestrito geraria mais de 100.000 colunas dispersas na matriz, o que sobrecarregaria a memória RAM do computador e causaria *overfitting* grave.

A estratégia adotada seleciona as 30 publicadoras e desenvolvedoras mais expressivas da Steam (como *Ubisoft, EA* e grandes estúdios *indie*) e mapeia todas as milhares de empresas menores sob o grupo `'Outros'`. O ajuste foi feito unicamente sobre o `X_train` para evitar vazamento de dados, resultando em 60 variáveis binárias.

### Tratamento de Dados Ausentes e Variáveis Numéricas

In [ ]:
#=========================================================
#   IMPUTAÇÃO DE NULOS E ESCALONAMENTO DE RECURSOS
#=========================================================
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Função para converter a faixa de proprietários no ponto médio
def extrair_midpoint_owners(coluna):
    def parse_faixa(val):
        if pd.isna(val) or not isinstance(val, str):
            return np.nan
        partes = val.split('-')
        if len(partes) == 2:
            try:
                low = float(partes[0].strip().replace(',', ''))
                high = float(partes[1].strip().replace(',', ''))
                return (low + high) / 2.0
            except ValueError:
                return np.nan
        return np.nan
    return coluna.apply(parse_faixa)

# Criação da coluna 'Estimated_Owners_Midpoint'
X_train['Estimated_Owners_Midpoint'] = extrair_midpoint_owners(X_train['Estimated owners'])
X_test['Estimated_Owners_Midpoint'] = extrair_midpoint_owners(X_test['Estimated owners'])

# Definição das colunas numéricas
colunas_numericas = ['Price', 'Achievements', 'Release_Year', 'Estimated_Owners_Midpoint']

# Imputação de Ausentes
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train[colunas_numericas])
X_test_imp = imputer.transform(X_test[colunas_numericas])

# Escalonamento (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Atualiza o X_train e X_test originais sem valores ausentes e com os dados normalizados
X_train[colunas_numericas] = pd.DataFrame(X_train_scaled, columns=colunas_numericas, index=X_train.index)
X_test[colunas_numericas] = pd.DataFrame(X_test_scaled, columns=colunas_numericas, index=X_test.index)

# Confirmação de que não restam valores nulos nas colunas numéricas
print("Valores nulos remanescentes no Treino:")
print(X_train[colunas_numericas].isna().sum())

print("Estatísticas do Treino após inclusão de Estimated Owners e escalonamento:")
print(X_train_num_scaled.describe().round(2))

**Interpretação:**

A variável `Estimated owners` continha intervalos em formato textual (ex: `'20000 - 50000'`). Para reaproveitar a hierarquia inerente a esses dados de forma quantitativa sem inflar a dimensão do dataset com *One-Hot Encoding*, transformou-se cada faixa no seu **ponto médio numérico** (`Estimated_Owners_Midpoint`).

Em seguida, aplicou-se a imputação por mediana e a padronização pelo `StandardScaler` junto a `Price`, `Achievements` e `Release_Year`. Isso permite que o `SGDClassifier` e a `RandomForestClassifier` capturem o impacto do volume estimado de proprietários como um preditor contínuo e escalonado.

### Consolidação Final de Todos os Atributos Processados

In [ ]:
#=========================================================
#   CONSOLIDAÇÃO FINAL DO DATASET DE TREINO E TESTE
#=========================================================

# Unificação de todos os atributos processados:
# - Categóricos: Gêneros, Categorias, Plataformas
# - Desenvolvedoras e Publicadoras
# - Numéricos Imputados e Escalonados

X_train_processed = pd.concat([
    X_train_membroA,
    devs_train_df,
    pubs_train_df,
    X_train_num_scaled
], axis=1)

X_test_processed = pd.concat([
    X_test_membroA,
    devs_test_df,
    pubs_test_df,
    X_test_num_scaled
], axis=1)

print(f"Formato final de X_train_processed: {X_train_processed.shape}")
print(f"Formato final de X_test_processed:  {X_test_processed.shape}")
print(f"Total de atributos preditivos na pipeline: {X_train_processed.shape[1]}")
X_train_processed.head()

**Interpretação:**

A matriz final de modelagem passa a contar com exatamente **158 atributos preditivos** (94 do Membro A + 60 do One-Hot de Devs/Publishers + 4 numéricos escalonados). Todas as colunas orfãs e textos brutos de `df_model` foram devidamente traduzidos e processados, finalizando o pré-processamento de forma robusta e livre de *data leakage*.

## Documentação e Justificativas Técnicas do Pipeline

Esta seção formaliza o embasamento teórico para as decisões de engenharia de dados adotadas na Etapa 2:

1. **Separação Treino/Teste e Estratificação (`stratify=y`):**
   Garante que a proporção da classe minoritária (`Success` ~10,4%) seja exatamente idêntica nos subconjuntos de treino e teste. Sem a estratificação, amostragens aleatórias simples em datasets desbalanceados correm o risco de concentrar exemplos positivos em um único conjunto, distorcendo os hiperparâmetros e as métricas do modelo.

2. **Isolamento de `X_test` para Evitar Vazamento de Dados (*Data Leakage*):**
   Todos os transformadores do `scikit-learn` (`MultiLabelBinarizer`, `SimpleImputer` e `StandardScaler`), assim como a seleção das empresas *Top N*, executaram o ajuste de hiperparâmetros (`fit`) **estritamente em `X_train`**. O conjunto de teste sofreu apenas a aplicação (`transform` / `reindex`), garantindo a integridade científica da avaliação final.

3. **Abordagem para Alta Cardinalidade (`Developers` / `Publishers`):**
   Em vez de eliminar as colunas ou aplicar *One-Hot Encoding* ingênuo (que geraria >130.000 atributos dispersos), agrupou-se as empresas no formato *Top 30 + Outros*. Isso permitiu capturar o efeito do prestígio das maiores publicadoras da indústria nos jogos sem inflar desnecessariamente a dimensionalidade do modelo.

4. **Codificação Multi-Label para `Genres` e `Categories`:**
   Diferente do *One-Hot Encoding* padrão (que exige categorias mutuamente exclusivas), o `MultiLabelBinarizer` respeita a estrutura real do catálogo da Steam, permitindo que um mesmo jogo pertença simultaneamente a múltiplos gêneros (*RPG, Action, Indie*) através de vetores binários independentes.

5. **Tratamento de Escala para Modelos Lineares Baseados em Gradiente:**
   A padronização z-score das variáveis numéricas garante que atributos contínuos compartilhem a mesma escala (média 0 e variância 1). Isso otimiza as superfícies de custo do `SGDClassifier`, evitando que variáveis com maior magnitude absoluta dominem indevidamente a atualização dos pesos do modelo.

# 5: Modelagem e Validação

Esta fase é dedicada à construção e otimização dos algoritmos de aprendizado de máquina para prever o sucesso dos jogos na Steam. O fluxo de trabalho foi estruturado da seguinte maneira:
*   Modelo Baseline: Estabelecimento de um classificador de referência utilizando regras ingênuas para criar uma linha de base de desempenho.
*   Treinamento de Modelos: Implementação dos classificadores exigidos para a tarefa: SGDClassifier e RandomForestClassifier.
*   Otimização e Validação: Aplicação de validação cruzada exclusivamente no conjunto de treinamento e busca automatizada pelos melhores hiperparâmetros de cada algoritmo.

### MODELO BASELINE

In [ ]:
#==================================
#   MODELO BASELINE
#==================================

# Importação do modelo baseline e das métricas
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Instanciando o DummyClassifier com a estratégia de prever a classe mais frequente
baseline_model = DummyClassifier(strategy="most_frequent")

# Treinando o modelo com os dados de treino definidos na Etapa 2
baseline_model.fit(X_train, y_train)

# Realizando previsões no conjunto de teste para avaliação
y_pred_baseline = baseline_model.predict(X_test)

# Avaliação rápida do Baseline
print("--- RESULTADOS DO MODELO BASELINE ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_baseline):.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred_baseline, zero_division=0))

### Implementação do SGDClassifier

A etapa seguinte consiste na implementação do classificador SGD (Stochastic Gradient Descent). Para garantir que o modelo aprenda os padrões dos dados de forma robusta e não apenas memorize as informações, o treinamento é realizado utilizando validação cruzada no conjunto de treino. Além disso, aplicamos uma busca automatizada para explorar e encontrar a melhor combinação de hiperparâmetros para o algoritmo, testando diferentes configurações para a função de perda (loss), a penalidade de regularização (penalty) e a constante de regularização (alpha). O modelo final otimizado é então avaliado no conjunto de teste.

In [ ]:
#==================================
#   MODELO SGDCLASSIFIER E OTIMIZAÇÃO
#==================================

from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Definindo o modelo base SGD com um estado aleatório fixo para reprodutibilidade
sgd_base = SGDClassifier(random_state=42)

# Definindo a grade de hiperparâmetros exigida para exploração
param_grid_sgd = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],
    'penalty': ['l2', 'l1', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.01, 0.1]
}

# Configurando a busca em grade com validação cruzada de 5 partições (cv=5)
# Otimizando com base na métrica f1_macro devido ao desbalanceamento das classes
grid_sgd = GridSearchCV(
    estimator=sgd_base,
    param_grid=param_grid_sgd,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Treinando o modelo e buscando os melhores hiperparâmetros nos dados processados
print("Iniciando o treinamento e otimização do SGDClassifier...")
grid_sgd.fit(X_train_processed, y_train)

# Extraindo e exibindo o melhor modelo e seus hiperparâmetros
best_sgd = grid_sgd.best_estimator_
print("\nMelhores hiperparâmetros encontrados:")
print(grid_sgd.best_params_)

# Realizando as previsões no conjunto de teste isolado e processado
y_pred_sgd = best_sgd.predict(X_test_processed)

# Gerando as métricas de avaliação
print("\n--- RESULTADOS DO SGDCLASSIFIER OTIMIZADO ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_sgd):.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred_sgd, zero_division=0))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_sgd))

### MODELO RANDOM FOREST E OTIMIZAÇÃO

In [ ]:
#==================================
#   MODELO RANDOM FOREST E OTIMIZAÇÃO
#==================================

from sklearn.ensemble import RandomForestClassifier

# Definindo o modelo base Random Forest com estado aleatório fixo
rf_base = RandomForestClassifier(random_state=42)

# Definindo a grade de hiperparâmetros para exploração
# Limitamos as opções para que o tempo de processamento não seja excessivo
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Configurando a busca em grade com validação cruzada (cv=5)
grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Treinamento e busca dos melhores hiperparâmetros
print("Iniciando o treinamento e otimização do RandomForestClassifier...")
grid_rf.fit(X_train_processed, y_train)

# Extraindo e exibindo o melhor modelo e seus hiperparâmetros
best_rf = grid_rf.best_estimator_
print("\nMelhores hiperparâmetros encontrados:")
print(grid_rf.best_params_)

# Realizando previsões no conjunto de teste
y_pred_rf = best_rf.predict(X_test_processed)

# Gerando as métricas de avaliação[cite: 1]
print("\n--- RESULTADOS DO RANDOM FOREST OTIMIZADO ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred_rf, zero_division=0))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_rf))

#==================================
#   IMPORTÂNCIA DAS CARACTERÍSTICAS
#==================================

# Extraindo as importâncias geradas pela floresta
importancias = best_rf.feature_importances_
nomes_features = X_train_processed.columns

# Criando um DataFrame para facilitar a visualização e pegando o Top 15
df_importancias = pd.DataFrame({'Feature': nomes_features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False).head(15)

# Plotando o gráfico de importância
plt.figure(figsize=(12, 8))
sns.barplot(x='Importancia', y='Feature', data=df_importancias, palette='viridis')
plt.title('Top 15 Características Mais Importantes para o Sucesso - Random Forest')
plt.xlabel('Grau de Importância')
plt.ylabel('Atributo')
plt.tight_layout()
plt.show()

Iniciando o treinamento e otimização do RandomForestClassifier...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


### Discussão Crítica dos Resultados

* **Comparação de Modelos:** O **RandomForestClassifier** apresentou o melhor desempenho geral para o objetivo do projeto. Ele alcançou um **F1-Score de 0.48** para a classe "Sucesso", superando expressivamente o **SGDClassifier** (0.31) e o modelo **Baseline** (0.00).


* **Motivo da Escolha:** A escolha do Random Forest se justifica pela sua capacidade de modelar **relações não-lineares** complexas entre as variáveis e lidar melhor com o **desbalanceamento de classes**. Sua **precisão de 71%** demonstra que, quando o modelo prevê que um jogo será um sucesso, ele tem uma alta taxa de confiabilidade.


* **Erros Observados:** O principal erro diagnóstico, visível na **Matriz de Confusão**, é o alto volume de **Falsos Negativos** (1663 no Random Forest e 2053 no SGD). Os modelos são conservadores e acabam classificando muitos jogos de sucesso como "Não Sucesso", o que reflete um **recall baixo**.


* **Limitações do Modelo:** A maior limitação enfrentada é estrutural: o **extremo desbalanceamento** do dataset (proporção de 9 para 1). A máquina possui poucos exemplos da classe minoritária para extrair padrões definitivos de sucesso. Além disso, a métrica de sucesso é complexa e subjetiva, dependendo de fatores externos de marketing e comunidade que não estão mapeados nas características técnicas do dataset.


* **Possíveis Melhorias:** Havendo mais tempo de projeto, o desempenho preditivo poderia ser aprimorado com o uso de técnicas de **balanceamento de classes**, como *oversampling* (ex: SMOTE) nos dados de treino. Também seria proveitoso testar **outros modelos**, como Gradient Boosting (XGBoost ou LightGBM), e realizar uma **busca de hiperparâmetros** mais longa e exaustiva para tentar elevar a taxa de captura (*recall*).